When looking at the coefficient of birth year, the age window 55-60 showed extreme values in every region. this is a debug to see if the age window is valid (contains any outliers)

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from tqdm import tqdm

combined_df = pd.read_pickle('/home/gaia/Projects/legacy_data/combined_gm_volumes.pkl')
 
# keep only classification_label=1 and snbb
volumes = combined_df[(combined_df['classification_label'] == 1) | (combined_df['source'] == 'snbb')]

In [ ]:
print(volumes.columns)

: 

In [ ]:
late_fifties_df = volumes[(volumes['age_in_years'] >= 55) & (volumes['age_in_years'] < 60)].copy()

subjects_late_fifties = late_fifties_df['subject_id'].unique()
print(f"Number of subjects in late fifties: {len(subjects_late_fifties)}")

# compare late fifties to early thirties 
early_thirties_df = volumes[(volumes['age_in_years'] >= 30) & (volumes['age_in_years'] < 35)].copy()
subjects_early_thirties = early_thirties_df['subject_id'].unique()
print(f"Number of subjects in early thirties: {len(subjects_early_thirties)}")

# define relevant regions for testing
random_regions = [421]  # for consistent results during testing

# generate n random regions between 1 to 454
region_list = np.arange(1, 455)
n = 5
random_regions = np.random.choice(region_list, n, replace=False)

: 

In [ ]:
# plot tiv distribution
plt.figure(figsize=(10, 6))
sns.histplot(early_thirties_df['tiv'], kde=True, color='orange')
sns.histplot(late_fifties_df['tiv'], kde=True, color='blue')
plt.title('Distribution of Total Intracranial Volume (TIV) in Early Thirties and Late Fifties')
plt.xlabel('TIV (mm³)')
plt.ylabel('Frequency')
plt.show()


# plot volume_mm3 distribution for random regions, violin plot
for region in random_regions:
    region_early_df = early_thirties_df[early_thirties_df['region_label'] == region]
    region_late_df = late_fifties_df[late_fifties_df['region_label'] == region]
    plt.figure(figsize=(10, 6))
    sns.violinplot(x=region_early_df['volume_mm3'], color='orange', label='Early Thirties')
    sns.violinplot(x=region_late_df['volume_mm3'], color='blue', label='Late Fifties')
    plt.title(f'Volume Distribution for Region {region} in Early Thirties and Late Fifties')
    plt.xlabel('Volume (mm³)')
    plt.legend()
    plt.show()

: 

In [ ]:
import re
import seaborn as sns
import matplotlib.pyplot as plt

# --- 1️⃣ Normalize region names to a common base (remove LH/RH indicators) ---
def extract_base_region(name):
    # Remove "_LH_" / "_RH_" in middle or "-lh" / "-rh" at end
    name = re.sub(r'_[LR]H_', '_', name)
    name = re.sub(r'[-_](lh|rh)$', '', name, flags=re.IGNORECASE)
    return name

significant_results_df['region_base'] = significant_results_df['region_name'].apply(extract_base_region)

# --- 2️⃣ Define consistent colors for hemispheres ---
hemi_colors = {
    'LH': '#8DD3C6',  # mint green
    'RH': '#FF5921',  # orange-red
    'Unknown': 'gray'
}

# --- 3️⃣ Loop over significant regions and plot ---
for base_name, df_group in significant_results_df.groupby('region_base'):
    if not any(df_group['variable'] == 'birth_year'):
        continue

    roi_labels = df_group['region_label'].tolist()
    df_plot = volumes[volumes['region_label'].isin(roi_labels)].copy()

    # Identify hemisphere for each ROI
    df_plot['hemisphere'] = df_plot['region_label'].map({
        row['region_label']: (
            'LH' if re.search(r'(_LH_|-lh$)', row['region_name'], flags=re.IGNORECASE) else
            'RH' if re.search(r'(_RH_|-rh$)', row['region_name'], flags=re.IGNORECASE) else
            'Unknown'
        )
        for _, row in df_group.iterrows()
    })

    # --- 4️⃣ Plot: LH and RH in fixed colors ---
    plt.figure(figsize=(4, 3))
    sns.scatterplot(
        data=df_plot, x='birth_year', y='volume_mm3',
        hue='hemisphere', hue_order=['LH', 'RH'] , palette=hemi_colors, alpha=0.6, s=25
    )

    # Add separate regression lines with fixed color
    for hemi in ['LH', 'RH']:
        hemi_data = df_plot[df_plot['hemisphere'] == hemi]
        if not hemi_data.empty:
            sns.regplot(
                data=hemi_data, x='birth_year', y='volume_mm3',
                scatter=False, line_kws={'color': hemi_colors[hemi], 'lw': 2}
            )

    plt.title(f"{base_name}")
    plt.xlabel("Birth Year")
    plt.ylabel("Gray Matter Volume (mm³)")
    plt.legend(title="Hemisphere")
    plt.tight_layout()
    plt.show()


: 

In [ ]:
late_fifties_df.describe()

: 

In [ ]:
early_thirties_df.describe()

: 

# coefficiant distributions

In [ ]:
coef_df = pd.read_csv("/home/gaia/Projects/legacy_data/legacy_pipe/data/interim/coef_df_age_bins_size_5.csv")

: 